# Whisper + YOLO 병합

**목표:** Whisper 타임스탬프 기준으로 YOLO 객체를 병합하여 통합 CSV 생성

**입력:**
- `whisper_metadata.csv` — Whisper 대사 + 타임스탬프 (수정본)
- `Yolo_scene_objects.csv` — YOLO 씬별 객체 탐지 결과

**출력:**
- `merged_whisper_yolo.csv` — 병합 결과 (Gemini 분석 입력용)

## 0. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive 마운트 완료!')

## 1. 파일 로드 & 컬럼 확인

In [ ]:
import pandas as pd

BASE = '/content/drive/MyDrive/SampleVideo.Test'

df_whisper = pd.read_csv(f'{BASE}/whisper_metadata.csv')
df_yolo    = pd.read_csv(f'{BASE}/Yolo_scene_objects.csv')

print('=== whisper_metadata.csv ===')
print('컬럼:', df_whisper.columns.tolist())
print(f'행 수: {len(df_whisper)}')
print(df_whisper.head(3))

print('\n=== Yolo_scene_objects.csv ===')
print('컬럼:', df_yolo.columns.tolist())
print(f'행 수: {len(df_yolo)}')
print(df_yolo.head(3))

## 2. 컬럼 매핑 설정

위 출력 결과를 보고 컬럼명이 다르면 아래 셀에서 직접 수정하세요.

In [ ]:
# ============================================================
# 컬럼 자동 감지
# ============================================================
def _col(df, *candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f'컬럼 없음. 후보={candidates}  실제={df.columns.tolist()}')

# Whisper
W_FILENAME = _col(df_whisper, 'Filename', 'filename', 'clip_name', 'file_name')
W_START    = _col(df_whisper, 'Start Time(s)', 'start_sec', 'start_time', 'start')
W_END      = _col(df_whisper, 'End Time(s)',   'end_sec',   'end_time',   'end')
W_DIALOGUE = _col(df_whisper, 'Context Text',  'dialogue',  'text', 'transcript', 'context_text')

# YOLO
Y_START   = _col(df_yolo, 'start_sec', 'Start Time(s)', 'start_time', 'start')
Y_END     = _col(df_yolo, 'end_sec',   'End Time(s)',   'end_time',   'end')
Y_OBJECTS = _col(df_yolo, 'detected_objects', 'objects', 'labels', 'object_names')

print('컬럼 매핑 확인:')
print(f'  Whisper → filename:{W_FILENAME}  start:{W_START}  end:{W_END}  dialogue:{W_DIALOGUE}')
print(f'  YOLO    → start:{Y_START}  end:{Y_END}  objects:{Y_OBJECTS}')

## 3. 타임스탬프 기준 병합

In [ ]:
from tqdm.auto import tqdm

# YOLO 타임스탬프 숫자형 변환
df_yolo[Y_START] = pd.to_numeric(df_yolo[Y_START], errors='coerce')
df_yolo[Y_END]   = pd.to_numeric(df_yolo[Y_END],   errors='coerce')

merged_rows = []

for idx, w_row in tqdm(df_whisper.iterrows(), total=len(df_whisper), desc='병합 중'):
    w_s = float(w_row[W_START])
    w_e = float(w_row[W_END])

    # Whisper 씬 구간과 겹치는 YOLO 행 찾기
    mask = (df_yolo[Y_START] < w_e) & (df_yolo[Y_END] > w_s)
    matched = df_yolo[mask]

    if len(matched) > 0:
        all_objects = []
        for objs in matched[Y_OBJECTS].dropna():
            all_objects.extend([o.strip() for o in str(objs).split(',') if o.strip()])
        detected = ', '.join(sorted(set(all_objects))) if all_objects else '(탐지 없음)'
    else:
        detected = '(탐지 없음)'

    merged_rows.append({
        'context_num':      idx + 1,
        'filename':         str(w_row[W_FILENAME]),
        'start_sec':        w_s,
        'end_sec':          w_e,
        'dialogue':         str(w_row[W_DIALOGUE]) if pd.notna(w_row[W_DIALOGUE]) else '(대사 없음)',
        'detected_objects': detected,
    })

df_merged = pd.DataFrame(merged_rows)

print(f'\n병합 완료! {len(df_merged)}개 씬')
print(f'탐지 없음: {(df_merged["detected_objects"] == "(탐지 없음)").sum()}건')
print(df_merged.head(5))

## 4. 결과 저장

In [ ]:
import shutil

output_name = 'merged_whisper_yolo.csv'

# 로컬 저장
df_merged.to_csv(output_name, index=False, encoding='utf-8-sig')

# Drive 백업
drive_path = f'{BASE}/{output_name}'
shutil.copy(output_name, drive_path)
print(f'Drive 저장 완료: {drive_path}')

# 로컬 다운로드
from google.colab import files
files.download(output_name)
print('다운로드 완료!')

## 5. 병합 결과 검증

In [ ]:
print('=== 병합 결과 검증 ===')
print(f'총 씬 수:         {len(df_merged)}')
print(f'start_sec 0인 행: {(df_merged["start_sec"] == 0).sum()}건')
print(f'탐지 없음:        {(df_merged["detected_objects"] == "(탐지 없음)").sum()}건')
print(f'대사 없음:        {(df_merged["dialogue"] == "(대사 없음)").sum()}건')

print('\n--- 타임스탬프 분포 ---')
print(df_merged[['start_sec', 'end_sec']].describe())

print('\n--- 샘플 5건 ---')
print(df_merged[['context_num','filename','start_sec','end_sec','dialogue','detected_objects']].head(5).to_string())